# Egyptian ID — Rotation + Verification + OCR Pipeline

**Flow:**
- **Step 0** – Auto-rotation (rembg + dimension + green-strip flip check)
- **Steps 1-2** – Load real/fake reference images (cached)
- **Step 3** – Dual-reference verification gate
- **Step 4** – Front-ID OCR (name, address, ID number, birthdate)
- **Step 5** – Back-ID OCR (expiration date)


In [ ]:
# ============================================================================
# Egyptian ID — Full Verification + OCR Pipeline
# ============================================================================
#
#  FLOW:
#   1. Select real reference images  (1–3 genuine IDs)
#   2. Select fake reference images  (1–10 AI/printed fakes, optional)
#   3. Select FRONT of test ID
#        → Dual-reference verification gate
#        → FAKE?  →  stop, print rejection
#        → PASS?  →  run front-ID OCR → extract name, address, national ID, birthdate
#   4. Select BACK of test ID
#        → OCR → extract expiration date
#   5. Print final combined data
#
# ============================================================================

# ─── Imports ─────────────────────────────────────────────────────────────────

from tkinter import *
import cv2
import re
import os
import sys
from tkinter import filedialog
from rembg import remove as rembg_remove
from PIL import Image
from easyocr import easyocr
import numpy as np
import pytesseract
import string
from datetime import datetime
from skimage.metrics import structural_similarity as ssim

# ─── Tesseract paths ─────────────────────────────────────────────────────────
pytesseract.pytesseract.tesseract_cmd = r'D:\ocr\tesseract\tesseract.exe'
os.environ['TESSDATA_PREFIX'] = r'D:\ocr\tessdata'

# ─── OCR data dict ───────────────────────────────────────────────────────────
data = {"first name": "0",
        "seconed name": "0",
        "address": "0",
        "id": "0",
        "birthdate": "0",
        "Expiration date": None,
        "error": 0}

# ─── OCR configuration ───────────────────────────────────────────────────────
TESS_LANG_TEXT = "ara"
TESS_LANG_ID   = "ara"
TESS_CONFIG_TEXT = "--psm 11 --oem 3"
TESS_CONFIG_ID   = "--psm 7 --oem 3"

ARABIC_DIGITS = ["٠", "١", "٢", "٣", "٤", "٥", "٦", "٧", "٨", "٩"]
PUN = set(string.punctuation)
_ARABIC_TO_WESTERN = str.maketrans('٠١٢٣٤٥٦٧٨٩', '0123456789')
_WESTERN_TO_ARABIC = str.maketrans('0123456789', '٠١٢٣٤٥٦٧٨٩')

# ─── Verification configuration ──────────────────────────────────────────────
OUTPUT_SIZE         = (640, 400)
PASSING_SCORE       = 50.0
FAKE_MARGIN         = 0.0
MAX_REFERENCES      = 3
MAX_FAKE_REFERENCES = 10

ZONES = [
    (0.00, 0.22, 0.25, 1.00, 0.08, "header"),
    (0.25, 0.75, 0.30, 0.80, 0.30, "gold_center"),
    (0.80, 1.00, 0.00, 1.00, 0.30, "bottom_strip"),
    (0.00, 0.22, 0.00, 0.25, 0.20, "top_left"),
    (0.25, 0.90, 0.45, 1.00, 0.12, "guilloche_bg"),
]
ZONE_PRESENCE_THRESHOLD = 0.25

# Path where preprocessed references are cached between runs
REFS_SAVE_PATH = r"D:\ocr\references_cache.npz"

# ═══════════════════════════════════════════════════════════════════════════════
#  VERIFICATION FUNCTIONS  (from id_verification_v2.py)
# ═══════════════════════════════════════════════════════════════════════════════

def _find_card_corners(img_bgr):
    orig_h, orig_w = img_bgr.shape[:2]
    scale = min(1.0, 1000.0 / orig_w)
    work  = cv2.resize(img_bgr, (int(orig_w * scale), int(orig_h * scale)))
    try:
        pil_img   = Image.fromarray(cv2.cvtColor(work, cv2.COLOR_BGR2RGB))
        pil_out   = rembg_remove(pil_img)
        rgba      = np.array(pil_out)
        mask      = (rgba[:, :, 3] > 10).astype(np.uint8) * 255
        card_work = cv2.cvtColor(rgba[:, :, :3], cv2.COLOR_RGB2BGR)
    except Exception:
        card_work = work
        mask = np.ones(work.shape[:2], dtype=np.uint8) * 255
    masked  = cv2.bitwise_and(card_work, card_work, mask=mask)
    gray    = cv2.cvtColor(masked, cv2.COLOR_BGR2GRAY)
    clahe   = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray    = clahe.apply(gray)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    edges   = cv2.Canny(blurred, 30, 120)
    kernel  = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    edges   = cv2.dilate(edges, kernel, iterations=2)
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None, scale
    contours  = sorted(contours, key=cv2.contourArea, reverse=True)
    work_area = work.shape[0] * work.shape[1]
    for cnt in contours[:5]:
        if cv2.contourArea(cnt) < work_area * 0.10:
            continue
        peri   = cv2.arcLength(cnt, True)
        approx = cv2.approxPolyDP(cnt, 0.02 * peri, True)
        if len(approx) == 4:
            return approx, scale
    x, y, w, h = cv2.boundingRect(contours[0])
    rect = np.array([[[x,y]],[[x+w,y]],[[x+w,y+h]],[[x,y+h]]], dtype=np.int32)
    return rect, scale


def _order_points(pts):
    pts  = pts.reshape(4, 2).astype(np.float32)
    rect = np.zeros((4, 2), dtype=np.float32)
    s    = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]; rect[2] = pts[np.argmax(s)]
    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]; rect[3] = pts[np.argmax(diff)]
    return rect


def _extract_card(original_bgr, corners_scaled, scale):
    pts   = _order_points(corners_scaled / scale)
    w1    = np.linalg.norm(pts[1]-pts[0]); w2 = np.linalg.norm(pts[2]-pts[3])
    h1    = np.linalg.norm(pts[3]-pts[0]); h2 = np.linalg.norm(pts[2]-pts[1])
    max_w = int(max(w1, w2)); max_h = int(max(h1, h2))
    if max_h > max_w: max_w, max_h = max_h, max_w
    dst   = np.array([[0,0],[max_w,0],[max_w,max_h],[0,max_h]], dtype=np.float32)
    M     = cv2.getPerspectiveTransform(pts, dst)
    return cv2.warpPerspective(original_bgr, M, (max_w, max_h))


def _remove_shadow(img_bgr):
    result_channels = []
    kh = max(10, img_bgr.shape[0]//5); kw = max(10, img_bgr.shape[1]//5)
    kh = kh if kh%2==1 else kh+1; kw = kw if kw%2==1 else kw+1
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kw, kh))
    for ch in cv2.split(img_bgr):
        bg   = cv2.morphologyEx(ch, cv2.MORPH_CLOSE, kernel)
        norm = (ch.astype(np.float32) / (bg.astype(np.float32)+1e-6)) * 200.0
        result_channels.append(np.clip(norm, 0, 255).astype(np.uint8))
    return cv2.merge(result_channels)


def _preprocess_card(img_bgr, label=""):
    corners, scale = _find_card_corners(img_bgr)
    card = _extract_card(img_bgr, corners.astype(np.float32), scale) if corners is not None else img_bgr.copy()
    if card.shape[0] > card.shape[1]:
        card = cv2.rotate(card, cv2.ROTATE_90_CLOCKWISE)
    card = _remove_shadow(card)
    lab     = cv2.cvtColor(card, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l       = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(8,8)).apply(l)
    card    = cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2BGR)
    return cv2.resize(card, OUTPUT_SIZE)


def _crop_zone(img, y1p, y2p, x1p, x2p):
    h, w = img.shape[:2]
    return img[int(h*y1p):int(h*y2p), int(w*x1p):int(w*x2p)]


def _zone_ssim(a, b):
    ga = cv2.cvtColor(a, cv2.COLOR_BGR2GRAY)
    gb = cv2.cvtColor(b, cv2.COLOR_BGR2GRAY)
    ms = min(ga.shape + gb.shape)
    win = min(7, ms if ms%2==1 else ms-1)
    if win < 3: return 0.0
    sc, _ = ssim(ga, gb, full=True, win_size=win)
    return max(0.0, float(sc))


def _zone_color(a, b):
    ha = cv2.cvtColor(a, cv2.COLOR_BGR2HSV)
    hb = cv2.cvtColor(b, cv2.COLOR_BGR2HSV)
    scores = []
    for ch in range(3):
        h1 = cv2.calcHist([ha],[ch],None,[32],[0,256]); cv2.normalize(h1,h1)
        h2 = cv2.calcHist([hb],[ch],None,[32],[0,256]); cv2.normalize(h2,h2)
        scores.append(max(0.0, float(cv2.compareHist(h1,h2,cv2.HISTCMP_CORREL))))
    return float(np.mean(scores))


def _compare_zone(test, ref, zone_def):
    y1p, y2p, x1p, x2p, w, name = zone_def
    zt = _crop_zone(test, y1p, y2p, x1p, x2p)
    zr = _crop_zone(ref,  y1p, y2p, x1p, x2p)
    if zt.size == 0 or zr.size == 0: return 0.0, False
    raw      = _zone_ssim(zt, zr)*0.55 + _zone_color(zt, zr)*0.45
    baseline = 1.0*0.55 + _zone_color(zr, zr)*0.45
    score    = min((raw / max(baseline, 0.01))*100.0, 100.0)
    return score, score >= ZONE_PRESENCE_THRESHOLD*100.0


def _score_against_set(test, ref_set, label="REF"):
    best_score = 0.0; best_name = ""
    for i, (ref_img, ref_name) in enumerate(ref_set):
        tw = ts = 0.0; rejects = []
        for zd in ZONES:
            y1p,y2p,x1p,x2p,weight,name = zd
            sc, present = _compare_zone(test, ref_img, zd)
            _, ref_pres = _compare_zone(ref_img, ref_img, zd)
            if ref_pres and not present: rejects.append(name)
            ts += sc * weight; tw += weight
        final = ts/tw if tw > 0 else 0.0
        final = max(0.0, final - len(rejects)*15.0)
        if final > best_score: best_score = final; best_name = ref_name
    return round(best_score, 1), best_name


def verify_image(test_img, real_refs, fake_refs):
    """Dual-reference classifier. Returns (is_authentic, real_score, fake_score)."""
    real_score, real_best = _score_against_set(test_img, real_refs, "REAL")
    fake_score, fake_best = _score_against_set(test_img, fake_refs, "FAKE") if fake_refs else (0.0, "—")

    print(f"\n{'─'*55}")
    print(f"  [ VERIFICATION RESULT ]")
    print(f"  Real score : {real_score:.1f}  (vs {real_best})")
    if fake_refs:
        print(f"  Fake score : {fake_score:.1f}  (vs {fake_best})")
        print(f"  Margin     : {real_score:.1f} − {fake_score:.1f} = {real_score-fake_score:.1f}  (need > {FAKE_MARGIN})")
    print(f"  Threshold  : {PASSING_SCORE}")
    print(f"{'─'*55}")

    if fake_refs and fake_score >= real_score - FAKE_MARGIN:
        print(f"  ❌ IMAGE REJECTED  (Δ={real_score-fake_score:.1f})")
        return False, real_score, fake_score
    elif real_score >= PASSING_SCORE:
        print(f"  ✅ VERIFIED — Proceeding to OCR...")
        return True, real_score, fake_score
    else:
        print(f"  ❌ IMAGE REJECTED — Score {real_score:.1f} below threshold {PASSING_SCORE}")
        return False, real_score, fake_score


def _load_refs(paths, label):
    refs = []
    for path in paths:
        img = cv2.imread(path)
        if img is None: continue
        name = os.path.basename(path)
        print(f"  Loading {label}: {name}")
        refs.append((_preprocess_card(img), name))
    return refs


def save_references(real_refs, fake_refs, save_path=REFS_SAVE_PATH):
    """
    Save preprocessed real and fake references to a .npz cache file.
    Next run can reload instantly without re-running rembg + corner detection.
    """
    arrays = {}
    # Real references
    for i, (img, name) in enumerate(real_refs):
        arrays[f"real_img_{i}"]  = img
        arrays[f"real_name_{i}"] = np.array([name])   # store name as 1-element array
    arrays["real_count"] = np.array([len(real_refs)])
    # Fake references
    for i, (img, name) in enumerate(fake_refs):
        arrays[f"fake_img_{i}"]  = img
        arrays[f"fake_name_{i}"] = np.array([name])
    arrays["fake_count"] = np.array([len(fake_refs)])

    np.savez_compressed(save_path, **arrays)
    print(f"  ✓ References saved to {save_path}")
    print(f"    Real: {len(real_refs)}  |  Fake: {len(fake_refs)}")


def load_saved_references(save_path=REFS_SAVE_PATH):
    """
    Load preprocessed references from .npz cache.
    Returns (real_refs, fake_refs) in the same format as _load_refs().
    """
    if not os.path.exists(save_path):
        return None, None
    data_np = np.load(save_path, allow_pickle=True)
    real_count = int(data_np["real_count"][0])
    fake_count = int(data_np["fake_count"][0])
    real_refs  = [(data_np[f"real_img_{i}"], str(data_np[f"real_name_{i}"][0]))
                  for i in range(real_count)]
    fake_refs  = [(data_np[f"fake_img_{i}"], str(data_np[f"fake_name_{i}"][0]))
                  for i in range(fake_count)]
    return real_refs, fake_refs


def _pick_files(title, multiple=True):
    w = Tk(); w.withdraw(); w.attributes('-topmost', True)
    if multiple:
        paths = filedialog.askopenfilenames(parent=w, title=title,
                    filetypes=[("Images","*.jpg *.jpeg *.png *.bmp")])
    else:
        p = filedialog.askopenfilename(parent=w, title=title,
                    filetypes=[("Images","*.jpg *.jpeg *.png *.bmp")])
        paths = [p] if p else []
    w.destroy()
    return list(paths)


# ═══════════════════════════════════════════════════════════════════════════════
#  OCR HELPER FUNCTIONS  (from ocr.ipynb)
# ═══════════════════════════════════════════════════════════════════════════════

def _to_western_digits(sval): return (sval or "").translate(_ARABIC_TO_WESTERN)
def _to_arabic_digits(sval):  return (sval or "").translate(_WESTERN_TO_ARABIC)
def _count_arabic_letters(sval): return len(re.findall(r'[\u0600-\u06FF]', sval or ""))
def _arabic_words(sval): return re.findall(r'[\u0600-\u06FF]{2,}', sval or "")
def _clean_name(sval):
    sval = re.sub(r'[^\u0600-\u06FF\s]', ' ', sval or "")
    return ' '.join(sval.split())

def _sanitize_addr(sval):
    sval = (sval or "").replace('؟',' ').replace('?',' ').replace('>',' ').replace('<',' ')
    sval = re.sub(r'[a-zA-Z]', ' ', sval)
    sval = re.sub(r'[^\u0600-\u06FF0-9٠-٩\s\-ـ]', ' ', sval)
    return ' '.join(sval.split())

def _extract_leading_number(sval):
    sval = _sanitize_addr(sval)
    m = re.match(r'^[0-9٠-٩]{1,6}', sval)
    return m.group(0) if m else ""

def _extract_locality_prefix(sval):
    sval = _sanitize_addr(sval)
    m = re.search(r'[مقكش]|[0-9٠-٩]', sval)
    prefix = sval[:m.start()] if m else sval
    prefix = re.sub(r'[^\u0600-\u06FF\s]', ' ', prefix)
    return ' '.join(prefix.split())

def _extract_longest_arabic_phrase(sval):
    sval = _sanitize_addr(sval)
    if not sval: return ""
    tmp = re.sub(r'[0-9٠-٩]', ' ', sval)
    tmp = re.sub(r'\b[مقكش]\b', ' ', tmp)
    tmp = re.sub(r'[\-ـ]', ' ', tmp)
    tmp = ' '.join(tmp.split())
    phrases = re.findall(r'[\u0600-\u06FF]{2,}(?:\s+[\u0600-\u06FF]{2,}){0,3}', tmp)
    if not phrases: return ""
    phrases.sort(key=lambda p: (_count_arabic_letters(p), len(p.split()), len(p)), reverse=True)
    return phrases[0]

def _extract_all_locality_parts(sval):
    sval = _sanitize_addr(sval)
    cleaned = re.sub(r'\b[مقكش]\s*[\-ـ:]?\s*[0-9٠-٩]+', ' ', sval)
    cleaned = re.sub(r'\b[0-9٠-٩]+\b', ' ', cleaned)
    cleaned = re.sub(r'\b[مقكش]\b', ' ', cleaned)
    cleaned = re.sub(r'[\-ـ]+', ' ', cleaned)
    cleaned = re.sub(r'[^\u0600-\u06FF\s]', ' ', cleaned)
    return ' '.join(cleaned.split()).strip()

def _extract_marker_number(sval, marker):
    sval = _sanitize_addr(sval)
    m = re.search(rf'(?:^|[\s\-ـ]){marker}\s*[\-ـ:]?\s*([0-9٠-٩]{{1,3}})', sval)
    return m.group(1) if m else ""

def _closest_number_after_marker(sval, marker):
    sval = _sanitize_addr(sval)
    marker_idx = -1
    for match in re.finditer(rf'(?:^|[\s\-ـ]){marker}(?:[\s\-ـ]|$)', sval):
        marker_idx = match.start() + (1 if match.group(0)[0] in ' \-ـ' else 0)
        break
    if marker_idx == -1: return ""
    best = None
    for m in re.finditer(r'[0-9٠-٩]{2,3}', sval):
        dist = abs(m.start() - marker_idx)
        if best is None or dist < best[0]: best = (dist, m.group(0))
    return best[1] if best else ""

def _best_number(num_t, num_e, all_t, all_e):
    candidates = [x for x in [num_t, num_e] if x]
    if not candidates: return ""
    best = sorted(candidates, key=len, reverse=True)[0]
    if len(best) >= 2: return best
    singles = [d for d in (all_t+all_e) if len(d)==1 and d!=best]
    return best + singles[0] if singles else best

def _extract_city_district(sval):
    known_cities = ['اكتوبر','القاهرة','الجيزة','الاسكندرية','الاسماعيلية',
        'بورسعيد','السويس','المنصورة','طنطا','الزقازيق','اسيوط','الفيوم',
        'بنها','دمياط','اسوان','الاقصر','قنا','سوهاج','المنيا','كفر الشيخ',
        'الدقهلية','الشرقية','الغربية','القليوبية','البحيرة','مطروح']
    sval = _sanitize_addr(sval)
    words = sval.split()
    for i, word in enumerate(words):
        for city in known_cities:
            if city in word or word in city:
                return ' '.join(words[i:])
    if len(words) >= 2: return ' '.join(words[-2:])
    elif len(words) == 1: return words[0]
    return ""

def _extract_area_name(sval, city):
    sval = _sanitize_addr(sval)
    if city: sval = sval.replace(city, ' ')
    sval = re.sub(r'\b[مقكش]\s*[\-ـ:]?\s*[0-9٠-٩]+', ' ', sval)
    sval = re.sub(r'\b[0-9٠-٩]+\b', ' ', sval)
    sval = re.sub(r'\b[مقكش]\b', ' ', sval)
    sval = re.sub(r'[\-ـ]+', ' ', sval)
    return ' '.join(sval.split()).strip()

def choose_address(addr_tesseract, addr_easyocr):
    addr_t = _sanitize_addr(addr_tesseract)
    addr_e = _sanitize_addr(addr_easyocr)
    if not addr_t and not addr_e: return "0"
    city_t = _extract_city_district(addr_t); city_e = _extract_city_district(addr_e)
    city   = city_t if _count_arabic_letters(city_t) >= _count_arabic_letters(city_e) else city_e
    area_t = _extract_area_name(addr_t, city); area_e = _extract_area_name(addr_e, city)
    area   = area_t if _count_arabic_letters(area_t) >= _count_arabic_letters(area_e) else area_e
    markers = {}
    for marker in ['م','ق','ك','ش']:
        m_t = _extract_marker_number(addr_t, marker)
        m_e = _extract_marker_number(addr_e, marker)
        if not m_t and not m_e: continue
        m2_t = _closest_number_after_marker(addr_t, marker)
        m2_e = _closest_number_after_marker(addr_e, marker)
        all_t = [_to_western_digits(x) for x in re.findall(r'[0-9٠-٩]+', addr_t)]
        all_e = [_to_western_digits(x) for x in re.findall(r'[0-9٠-٩]+', addr_e)]
        best  = _best_number(_to_western_digits(m2_t or m_t), _to_western_digits(m2_e or m_e), all_t, all_e)
        if len(best)==1:
            twos = [d for d in (all_t+all_e) if len(d)==2]
            if twos: best = twos[-1]
        if best: markers[marker] = _to_arabic_digits(best)
    if len(markers)==0:   result = f"{area} {city}".strip() or addr_e or addr_t
    elif len(markers)==1:
        mk,num = list(markers.items())[0]
        result = f"{area} {mk} {num} {city}".strip()
    elif len(markers)==2:
        items  = list(markers.items())
        result = f"{area} {items[0][0]} {items[0][1]} -{items[1][0]} {items[1][1]} {city}".strip()
    else:
        result = f"{area} {' -'.join(f'{k} {v}' for k,v in markers.items())} {city}".strip()
    lead_t = _extract_leading_number(addr_t); lead_e = _extract_leading_number(addr_e)
    lead   = lead_t if len(lead_t)>=len(lead_e) else lead_e
    if lead:
        lead = _to_arabic_digits(_to_western_digits(lead))
        if not result.startswith(lead): result = f"{lead} {result}".strip()
    result = re.sub(r'[a-zA-Z]', '', result)
    result = re.sub(r'[^\u0600-\u06FF0-9٠-٩\s\-ـ]', ' ', result)
    return ' '.join(result.split())

def _extract_birthdate_from_id(id_value):
    digits = re.sub(r'\D', '', _to_western_digits(str(id_value)))
    if len(digits) < 7: return "0"
    if digits[0] in ('2','3') and len(digits) >= 7:
        century = 1900 if digits[0]=='2' else 2000
        try:
            dt = datetime(century+int(digits[1:3]), int(digits[3:5]), int(digits[5:7]))
            return dt.strftime('%Y-%m-%d')
        except Exception: return "0"
    if len(digits) >= 6:
        yy = int(digits[0:2]); mm = int(digits[2:4]); dd = int(digits[4:6])
        century = 2000 if yy <= (datetime.now().year % 100) else 1900
        try:
            dt = datetime(century+yy, mm, dd)
            return dt.strftime('%Y-%m-%d')
        except Exception: return "0"
    return "0"

def arabic_to_english_numbers(text):
    return text.translate(str.maketrans('٠١٢٣٤٥٦٧٨٩','0123456789'))


# ═══════════════════════════════════════════════════════════════════════════════
#  FRONT-ID OCR  (from ocr.ipynb cell 2)
# ═══════════════════════════════════════════════════════════════════════════════

def run_front_ocr(file_path):
    """Extract name, address, ID number and birthdate from the FRONT of the ID."""
    global data
    try:
        input  = Image.open(file_path)
        output = rembg_remove(input)
        img_array = np.array(output)
        img   = cv2.cvtColor(img_array, cv2.COLOR_RGBA2BGR)
        blurred   = cv2.blur(img, (5,5))
        kernel    = np.array([[-1,-1,-1],[-1,9,-1],[-1,-1,-1]])
        sharpened = cv2.filter2D(blurred, -1, kernel)
        canny     = cv2.Canny(sharpened, 50, 200)
        pts = np.argwhere(canny > 0)
        if pts.size == 0:
            cropped = img
        else:
            y1,x1 = pts.min(axis=0); y2,x2 = pts.max(axis=0)
            cropped = img[y1:y2, x1:x2]

        w,h,c = cropped.shape
        o = int(w/2); i = int(h/2.5); n = int(h/6)
        cr          = cropped[n-13:i+15, o:]
        cropped_img = cropped[i+8:, o+10:]

        cr_height   = cr.shape[0]; split_point = int(cr_height * 0.52)
        names_region   = cr[0:split_point, :]
        address_region = cr[split_point:, :]
        address_region = cv2.GaussianBlur(address_region, (3,3), 0)
        address_region = cv2.convertScaleAbs(address_region, alpha=1.3, beta=10)

        cv2.imwrite("newimg.png",  names_region)
        cv2.imwrite("address.png", address_region)
        cv2.imwrite("id_card.png", cropped_img)

        text_names   = pytesseract.image_to_string(names_region, lang='ara', config='--psm 11 --oem 3')
        splited_names = text_names.split('\n')

        arabic_digits = ["٠","١","٢","٣","٤","٥","٦","٧","٨","٩"]
        pun = set(string.punctuation)

        s         = easyocr.Reader(['ar','ar'])
        d_names   = s.readtext(names_region, detail=0, text_threshold=0.18, width_ths=0.9, low_text=0.17)
        d_address = s.readtext(address_region, detail=0, text_threshold=0.15, width_ths=0.7, low_text=0.15, paragraph=True)

        state = 0
        if len(text_names.split('\n')) == 4:
            state = 1
            data["first name"]   = splited_names[0] if len(splited_names)>0 else "0"
            data["seconed name"] = splited_names[2] if len(splited_names)>2 else "0"
            address_easyocr      = ' '.join(d_address) if d_address else ""
            data["address"]      = choose_address("", address_easyocr)

            imgs = cv2.imread('id_card.png', 0)
            gauss = cv2.GaussianBlur(imgs, (7,7), 0)
            unsharp = cv2.addWeighted(imgs, 2, gauss, -1, 0)
            o2 = s.readtext(unsharp, detail=0, text_threshold=0.27, width_ths=0.8, low_text=0.008)
            if len(o2)==1:   data["id"] = o2[0]
            elif len(o2)==0: data["id"] = "0"
            else:            data["id"] = max(o2, key=len)
        else:
            state = 2
            imgs  = cv2.imread('id_card.png', 0)
            imgs  = cv2.medianBlur(imgs, 3)
            gauss = cv2.GaussianBlur(imgs, (5,5), 0)
            unsharp = cv2.addWeighted(imgs, 2.2, gauss, -1.2, 0)
            unsharp = cv2.convertScaleAbs(unsharp, alpha=1.4, beta=5)
            d = d_names
            address_easyocr = ' '.join(d_address) if d_address else ""
            for tok in d:
                if tok in arabic_digits: break
                my_data    = ','.join(d)
                split_list = my_data.split(',')
                data["first name"]   = split_list[0] if split_list else "0"
                data["seconed name"] = ','.join(split_list[1:]).replace(","," ").strip() if len(split_list)>1 else "0"
                data["address"]      = choose_address("", address_easyocr).replace("[","").replace("]","").replace("'","")
            o2 = s.readtext(unsharp, detail=0, text_threshold=0.25, width_ths=0.7, low_text=0.01, paragraph=False)
            if o2 is None or d is None: data["error"] = "1"
            elif len(o2)==1:   data["id"] = o2[0]
            elif len(o2)==0:   data["id"] = "0"
            else:              data["id"] = max(o2, key=len)

        if len(str(data["id"])) < 20: data["error"] = "1"
        for c in list(data["first name"]):
            if c in arabic_digits or c in pun:
                data["first name"] = data["first name"].replace(c, "")
        for c in list(data["seconed name"]):
            if c in arabic_digits or c in pun:
                data["seconed name"] = data["seconed name"].replace(c, "")
        if len(data["seconed name"]) <= len(data["first name"]):
            data["first name"], data["seconed name"] = data["seconed name"], data["first name"]

        first_parts  = [p for p in (data["first name"] or "").split() if p]
        second_parts = [p for p in (data["seconed name"] or "").split() if p]
        if len(first_parts) > 3:  data["error"] = "1"
        if len(second_parts) <= 1: data["error"] = "1"

        ar = list('ابتثجحخدذرزسشصضطظعغفقكلمنهوي')
        for c in list(str(data["id"])):
            if c in ar or c in pun:
                data["id"] = str(data["id"]).replace(c, "")

        arabic_string = str(data["id"])
        matches = re.findall(r'[٠-٩]+', arabic_string)
        matches.reverse()
        concatenated = ''.join(matches)
        if concatenated:
            data["id"] = int(concatenated.translate(_ARABIC_TO_WESTERN))

        data["birthdate"] = _extract_birthdate_from_id(data["id"])
        if data["birthdate"] == "0": data["error"] = "1"

    except Exception as e:
        print("Front OCR Error:", e)
        data['id'] = 0; data['error'] = 1; data["birthdate"] = "0"


# ═══════════════════════════════════════════════════════════════════════════════
#  BACK-ID OCR  (from ocr.ipynb cell 3)
# ═══════════════════════════════════════════════════════════════════════════════

def run_back_ocr(file_path):
    """Extract expiration date from the BACK of the ID."""
    try:
        input_image  = Image.open(file_path)
        output_image = rembg_remove(input_image)
        img_array    = np.array(output_image)

        if len(img_array.shape)==3 and img_array.shape[2]==4:
            img_proc = cv2.cvtColor(img_array, cv2.COLOR_RGBA2BGRA)
        else:
            img_proc = img_array

        if len(img_proc.shape)==3 and img_proc.shape[2]==4:
            alpha  = img_proc[:,:,3]
            coords = cv2.findNonZero(alpha)
            if coords is not None:
                x,y,w,h = cv2.boundingRect(coords)
                img_proc = img_proc[y:y+h, x:x+w]

        h, w = img_proc.shape[:2]
        start_y = int(h*0.45); end_y = int(h*0.60)
        cropped = img_proc if start_y >= h else img_proc[start_y:end_y, :]
        if cropped.size == 0: return None
        cropped = cropped[:, :int(cropped.shape[1]*0.55)]
        if cropped.size == 0: return None

        gray   = cv2.cvtColor(cropped, cv2.COLOR_BGR2GRAY) if len(cropped.shape)==3 else cropped
        gauss  = cv2.GaussianBlur(gray, (7,7), 0)
        sharp  = cv2.addWeighted(gray, 2, gauss, -1, 0)
        gauss2 = cv2.GaussianBlur(sharp, (7,7), 0)
        final  = cv2.addWeighted(sharp, 2, gauss2, -1, 0)

        reader = easyocr.Reader(['ar','ar'], gpu=False)
        results = reader.readtext(final, detail=0, text_threshold=0.27,
                                  width_ths=0.8, low_text=0.008, allowlist='٠١٢٣٤٥٦٧٨٩')
        print(f"Back OCR results: {results}")

        for text in results:
            text = ''.join(re.findall(r'[٠-٩]', text.strip()))
            if not text or len(text) < 6: continue
            text = arabic_to_english_numbers(text)
            if int(text[0]) < 2: text = text[1:]
            year      = text[:4]
            month_raw = text[5:7] if len(text)>6 else text[4:6]
            if int(month_raw) > 12: month_raw = month_raw[1]
            month = month_raw.zfill(2)
            day_raw = text[7:] if len(text)>7 else text[6:]
            if len(day_raw) > 2: day_raw = day_raw[1:]
            day = day_raw.zfill(2)
            return f"{year}/{month}/{day}"

        return None
    except Exception as e:
        print(f"Back OCR Error: {e}")
        return None

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  ROTATION FUNCTIONS  (from rotation_app.py)
#  Auto-detects and corrects ID card orientation before verification + OCR
# ═══════════════════════════════════════════════════════════════════════════════

import json as _json
from pathlib import Path as _Path
from rembg import new_session as _new_session

print("Loading rembg rotation model …")
_REM_SESSION = _new_session("u2net")
print("rembg ready.")

_DIMS_JSON     = _Path(r"d:\ocr\national_id_ocr\id_reference_dims.json")
_REF_CROP_PATH = _Path(r"d:\ocr\national_id_ocr\ref_crop_cache.jpg")
_BUILTIN_REF_W = 718
_BUILTIN_REF_H = 453
_GREEN_STRIP_H          = 0.22
_GREEN_STRIP_CX         = 0.50
_GREEN_STRIP_CW         = 0.55
_GREEN_STRIP_OFFSET_PCT = 0.15
_RATIO_TOL              = 0.08

def _order_points(pts):
    rect = np.zeros((4, 2), dtype="float32")
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]; rect[2] = pts[np.argmax(s)]
    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]; rect[3] = pts[np.argmax(diff)]
    return rect

def _align_by_min_area_rect(image_bgr, contour):
    rect = cv2.minAreaRect(contour)
    box  = cv2.boxPoints(rect)
    pts  = _order_points(box)
    tl, tr, br, bl = pts
    maxW = int(max(np.linalg.norm(br-bl), np.linalg.norm(tr-tl)))
    maxH = int(max(np.linalg.norm(tr-br), np.linalg.norm(tl-bl)))
    dst  = np.array([[0,0],[maxW-1,0],[maxW-1,maxH-1],[0,maxH-1]], dtype="float32")
    M    = cv2.getPerspectiveTransform(pts, dst)
    return cv2.warpPerspective(image_bgr, M, (maxW, maxH)), maxW, maxH

def _rot_remove_bg_and_crop(image_bgr):
    pil_in  = Image.fromarray(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
    pil_out = rembg_remove(pil_in, session=_REM_SESSION,
                           alpha_matting=True,
                           alpha_matting_foreground_threshold=240,
                           alpha_matting_background_threshold=10)
    rgba    = cv2.cvtColor(np.array(pil_out), cv2.COLOR_RGBA2BGRA)
    alpha   = rgba[:, :, 3]
    _, mask = cv2.threshold(alpha, 10, 255, cv2.THRESH_BINARY)
    kernel  = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
    mask    = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=3)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        h, w = image_bgr.shape[:2]
        return image_bgr, w, h
    cropped, w, h = _align_by_min_area_rect(image_bgr, max(contours, key=cv2.contourArea))
    return cropped, w, h

def _top_center_strip(img):
    h, w = img.shape[:2]
    top  = max(int(h * _GREEN_STRIP_H), 1)
    cx   = int(w * _GREEN_STRIP_CX) + int(w * _GREEN_STRIP_OFFSET_PCT)
    half = int(w * _GREEN_STRIP_CW / 2)
    return img[:top, max(cx-half,0):min(cx+half,w)]

def _green_strip_hist(img):
    strip = _top_center_strip(img)
    if strip.size == 0:
        return np.zeros((64, 1), np.float32)
    hsv  = cv2.cvtColor(strip, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, (30, 20, 30), (95, 255, 255))
    hist = cv2.calcHist([hsv], [0], mask, [64], [0, 180])
    cv2.normalize(hist, hist)
    return hist

def _green_strip_sim(a, b):
    return float(cv2.compareHist(_green_strip_hist(a), _green_strip_hist(b), cv2.HISTCMP_INTERSECT))

def _rotate90(img, times):
    times = times % 4
    if times == 0: return img
    return cv2.rotate(img, {1: cv2.ROTATE_90_COUNTERCLOCKWISE,
                             2: cv2.ROTATE_180,
                             3: cv2.ROTATE_90_CLOCKWISE}[times])

def _orientation_matches(w, h, rw, rh):
    same = (w >= h) == (rw >= rh)
    cr = max(w,h) / max(min(w,h), 1)
    rr = max(rw,rh) / max(min(rw,rh), 1)
    return same and abs(cr-rr)/max(rr, 1e-6) < _RATIO_TOL

def _make_builtin_ref():
    img = np.full((_BUILTIN_REF_H, _BUILTIN_REF_W, 3), (180, 180, 170), dtype=np.uint8)
    top  = int(_BUILTIN_REF_H * _GREEN_STRIP_H)
    cx   = int(_BUILTIN_REF_W * _GREEN_STRIP_CX) + int(_BUILTIN_REF_W * _GREEN_STRIP_OFFSET_PCT)
    half = int(_BUILTIN_REF_W * _GREEN_STRIP_CW / 2)
    img[:top, max(cx-half,0):min(cx+half,_BUILTIN_REF_W)] = (65, 130, 115)
    return img

def _load_rot_reference():
    if _DIMS_JSON.exists() and _REF_CROP_PATH.exists():
        dims = _json.loads(_DIMS_JSON.read_text())
        ref  = cv2.imread(str(_REF_CROP_PATH))
        if ref is not None:
            return ref, dims["width"], dims["height"]
    return _make_builtin_ref(), _BUILTIN_REF_W, _BUILTIN_REF_H

def auto_rotate_id(image_bgr):
    # Auto-detect and correct orientation. Returns (corrected_bgr, rotation_degrees, notes).
    ref_bgr, ref_w, ref_h = _load_rot_reference()
    cropped, cw, ch = _rot_remove_bg_and_crop(image_bgr)

    # Step 1 — dimension-based rotation
    if _orientation_matches(cw, ch, ref_w, ref_h):
        rot, after_dim = 0, cropped
        note = f"Dim match at 0deg ({cw}x{ch} vs ref {ref_w}x{ref_h})"
    else:
        rot = None
        for deg in (90, 180, 270):
            r = _rotate90(cropped, deg // 90)
            rh_r, rw_r = r.shape[:2]
            if _orientation_matches(rw_r, rh_r, ref_w, ref_h):
                rot, after_dim = deg, r
                note = f"Dim match at {deg}deg"
                break
        if rot is None:
            after_dim, rot = cropped, 0
            note = f"No dim match ({cw}x{ch}) — kept original"

    # Step 2 — green-strip flip check
    sim_n  = _green_strip_sim(after_dim, ref_bgr)
    flipped = cv2.rotate(after_dim, cv2.ROTATE_180)
    sim180  = _green_strip_sim(flipped, ref_bgr)
    if sim180 > sim_n:
        final = flipped
        rot   = (rot + 180) % 360
        note += f" | Green-flip applied (sim {sim180:.3f}>{sim_n:.3f})"
    else:
        final = after_dim
        note += f" | No flip (sim {sim_n:.3f})"

    print(f"  [Rotation] {note}")
    print(f"  [Rotation] Final rotation: {rot}deg")
    return final, rot, note

print("Rotation functions loaded.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  MAIN PIPELINE  (Rotation → Verification → Front OCR → Back OCR)
# ═══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":

    print(f"\n{'═'*60}")
    print("  EGYPTIAN ID — ROTATION + VERIFICATION + OCR PIPELINE")
    print(f"{'═'*60}")

    # ── Load references ───────────────────────────────────────────────────────
    real_refs = fake_refs = None
    if os.path.exists(REFS_SAVE_PATH):
        real_refs, fake_refs = load_saved_references()
        if real_refs is not None:
            print(f"\n  Loaded from cache: {len(real_refs)} real + {len(fake_refs)} fake refs.")

    if real_refs is None:
        print("\n  STEP 1 — Select 1-3 REAL reference ID images")
        real_paths = _pick_files("Select REAL reference images", multiple=True)
        if not real_paths:
            print("No real references. Exiting."); sys.exit()
        real_refs = _load_refs(real_paths[:MAX_REFERENCES], "REAL")
        if not real_refs:
            print("No valid real references. Exiting."); sys.exit()
        print("\n  STEP 2 — Select FAKE references (Cancel to skip)")
        fake_paths = _pick_files("Select FAKE reference images (optional)", multiple=True)
        fake_refs  = _load_refs(fake_paths[:MAX_FAKE_REFERENCES], "FAKE") if fake_paths else []
        save_references(real_refs, fake_refs)
        print("  References saved — will auto-load next run.")

    print(f"  References ready: {len(real_refs)} real + {len(fake_refs)} fake.")

    # ── Select FRONT image ────────────────────────────────────────────────────
    print(f"\n  STEP 3 — Select the FRONT of the ID")
    front_paths = _pick_files("Select FRONT of ID image", multiple=False)
    if not front_paths:
        print("No image selected. Exiting."); sys.exit()
    front_path = front_paths[0]
    print(f"\n  Image: {os.path.basename(front_path)}")

    # ── STEP 0: Auto-rotate ───────────────────────────────────────────────────
    print(f"\n{'─'*60}")
    print("  STEP 0 — Auto-rotating to correct orientation …")
    print(f"{'─'*60}")
    front_img_raw = cv2.imread(front_path)
    if front_img_raw is None:
        print(f"Cannot read: {front_path}"); sys.exit()
    front_img, rotation_applied, rotation_note = auto_rotate_id(front_img_raw)
    print(f"  Rotation done: {rotation_applied}deg")

    # ── Verification ──────────────────────────────────────────────────────────
    print(f"\n  Verifying …")
    front_proc  = _preprocess_card(front_img)
    is_auth, real_sc, fake_sc = verify_image(front_proc, real_refs, fake_refs)

    if not is_auth:
        print(f"\n{'═'*60}")
        print("  ACCESS DENIED — FAKE or SUSPICIOUS image.")
        print("  OCR will NOT run.")
        print(f"{'═'*60}")
        sys.exit()

    print(f"\n  Image VERIFIED. Starting OCR …\n{'═'*60}")

    # ── Front OCR ─────────────────────────────────────────────────────────────
    print("  Running front-ID OCR …")
    run_front_ocr(front_path)

    # ── Back OCR ──────────────────────────────────────────────────────────────
    print(f"\n  STEP 5 — Select the BACK of the ID (for expiry date)")
    back_paths = _pick_files("Select BACK of ID image", multiple=False)
    extracted_date = None
    if back_paths:
        print("  Running back-ID OCR …")
        extracted_date = run_back_ocr(back_paths[0])
        print(f"\n  Expiration date: {extracted_date}" if extracted_date else "\n  Could not extract expiry date.")
    else:
        print("  No back image — skipping expiry date.")

    data["Expiration date"] = [extracted_date] if extracted_date else None
    data["id's"] = {
        "first name":      data["first name"],
        "second name":     data["seconed name"],
        "address":         data["address"],
        "birthdate":       data["birthdate"],
        "id":              data["id"],
        "Expiration date": data["Expiration date"],
        "rotation":        f"{rotation_applied}deg",
        "error":           data["error"],
    }

    print(f"\n{'═'*60}")
    print("  FINAL RESULT")
    print(f"{'═'*60}")
    for k, v in data["id's"].items():
        print(f"  {k:<20}: {v}")
    print(f"{'═'*60}")